## Package setup

In [1]:
import logging
import os
from pathlib import Path

from flexsipp_railways.generate import graph_from_file, scenario_from_file
from flexsipp.graphs.fsipp import FSIPP

# Disable logging of the program in the notebook
os.environ["LOGLEVEL"] = "CRITICAL"

logging.basicConfig()
logging.root.setLevel(logging.INFO)
logging.basicConfig(level=logging.INFO)

logger = logging.getLogger('__main__')
logger.setLevel(os.environ.get("LOGLEVEL", logging.FATAL))

pybooklogger = logging.getLogger('pybook')
pybooklogger.setLevel(logging.DEBUG)

## Infrastructure setup
Calculate the layout of the Dutch railway system


In [ ]:
## This requires the confidential infrastructure in the Netherlands
layout_file = Path("../../data/railways/prorail/netherlands-schiphol.json")
layout = graph_from_file(layout_file)

## Scenario Setup, load all scenarios and setup experiments

In [3]:
basepath = Path("../../data/railways/case_study_scenarios")
scenario_files = ["2025-07-08_1.json", "2025-07-08_2.json", "2025-07-08_3.json", "2025-07-08_4.json"]

Compute the blocking times and create the initial unsafe intervals

In [4]:
tad_exp = scenario_from_file(basepath / scenario_files[0], layout)
tad_exp.process()

Function Scenario.__init__ took 23.2293 seconds


KeyboardInterrupt: 

# Experiment
Set the delay agent, which is train with ID 1867

In [ ]:
delay_agent = tad_exp.get_replanning_agent("1867")

Take the graph and filter out the delayed agent, considering its perspective

In [ ]:
graph = tad_exp.fsipp(delay_agent)

Calculate the simple heuristic of the shortest path from all nodes to the agent's destination

In [ ]:
heuristic = graph.calculate_heuristic(delay_agent.destination)

Filter nodes in the graph to only the section of the Dutch railway network between Rotterdam (`Rtd`) and Schiphol (`Shl`) that routes through Delft (`Dt`) and Leiden (`Ledn`) using the short names of these areas.

In [ ]:
allowed_dpt = {"Rtd", "Rmoa_Rtd", "Sdm", "Dt_Sdm", "Dtcp", "Dt", "Dt_Gv", "Gvmw", "Gv", "Laa", "Gvm", "Gvm_Ledn", "Ledn", "Hfd_Ledn", "Hfd", "Hfd_Shl", "Shl"}

filter_nodes = {node for name, node in graph.nodes.items() if name.split("|")[0] in allowed_dpt}

Set up the graph with all agents, convert to safe intervals

In [ ]:
flexSIPP = FSIPP(graph, heuristic, tad_exp.agents, filter_nodes=filter_nodes)

Run the search over the graph of ATFs

In [ ]:
result = flexSIPP.run_search(delay_agent.origin.name, delay_agent.destination.name, delay_agent.measures.start_time, redirect_stderr="stderr.txt")

### Results

Plot the resulting ATF representing the path of the delayed agent

In [ ]:
from matplotlib import pyplot as plt

fig, axs = plt.subplots(2, 1, figsize=(5, 10), sharex=True)
result.plot(axs[0], linestyle=3)
result.plot(axs[1], show_atf=False, show_total_delays=True, original_arrival_time=delay_agent.measures.start_time)

Show the tipping points

In [ ]:
tipping_points = result.find_tipping_points(tad_exp.agents)
tipping_points